# BƯỚC 2: CHỨNG MINH DỮ LIỆU UY TÍN 

File `aqi_hcm_daily.csv` bạn vừa tạo ở Bước 1 là data thật, nhưng **"thật" chưa đủ**. Notebook này kiểm tra 6 việc, mỗi việc trả lời 1 câu hỏi khác nhau.


In [1]:
import requests
import pandas as pd
import numpy as np

DATA = "../01_Data/aqi_hcm_daily.csv"
LAT, LON = 10.78533, 106.67029          # dung lai toa do tram CMT8, de goi CAMS o Truc cot 4

df = pd.read_csv(DATA, parse_dates=["date"])   # doc lai dung file vua luu o Buoc 1
n = len(df)
print(f"Da doc {n} dong tu {DATA}")
print(f"Khoang thoi gian: {df['date'].min().date()} -> {df['date'].max().date()}")

Da doc 629 dong tu ../01_Data/aqi_hcm_daily.csv
Khoang thoi gian: 2024-11-19 -> 2026-09-21


## Bằng chứng 1: Nguồn gốc minh bạch 

**Câu hỏi trả lời:** data này lấy từ đâu, ai xác nhận lại được?

**Bản chất:** 1 dữ liệu đáng tin trước hết phải **truy được nguồn**.

In [2]:
print("[1] NGUON GOC MINH BACH")
print("-" * 60)
# Chi in ra thong tin tinh (khong tinh toan gi) -> muc dich la de nguoi doc
# (thay/ban hoc khac) co dia chi ro rang de tu di kiem tra lai neu muon
print("Nguon 1: OpenAQ - to chuc phi loi nhuan quoc te")
print("         Tram CMT8, Location ID 3276359, TP.HCM")
print("         Kiem chung: https://explore.openaq.org/locations/3276359")
print("Nguon 2: Open-Meteo Archive API - nguon ECMWF/CAMS (chau Au)")
print("=> Ai cung mo link kiem tra duoc -> minh bach, truy xuat duoc")

[1] NGUON GOC MINH BACH
------------------------------------------------------------
Nguon 1: OpenAQ - to chuc phi loi nhuan quoc te
         Tram CMT8, Location ID 3276359, TP.HCM
         Kiem chung: https://explore.openaq.org/locations/3276359
Nguon 2: Open-Meteo Archive API - nguon ECMWF/CAMS (chau Au)
=> Ai cung mo link kiem tra duoc -> minh bach, truy xuat duoc


## Bằng chứng 2: Chất lượng đo lường

**Câu hỏi trả lời:** data có bị thiếu nhiều không, có giá trị bất thường (outlier) không?

**Cách tính outlier — quy tắc IQR (khoảng tứ phân vị):**
- `Q1` = giá trị tại mốc 25% (25% số ngày có PM2.5 thấp hơn mốc này)
- `Q3` = giá trị tại mốc 75%
- `IQR = Q3 - Q1` = độ trải rộng của phần "giữa" dữ liệu (bỏ 25% thấp nhất và 25% cao nhất)
- Ngưỡng bất thường: thấp hơn `Q1 - 1.5*IQR` hoặc cao hơn `Q3 + 1.5*IQR` → coi là outlier

**Analogy:** giống đo chiều cao cả lớp, nếu 1 bạn cao 2.3m thì lệch hẳn khỏi khoảng chiều cao thường thấy của đa số — số 1.5 lần IQR là ngưỡng "lệch hẳn" được thống kê học quy ước dùng phổ biến.

In [4]:
print("[2] CHAT LUONG DO LUONG")
print("-" * 60)

cov = df["pm25_cov"]
print(f"Do phu TB (% gio/ngay co do)   : {cov.mean():.1f}%")
print(f"So ngay phu >= 75%             : {(cov >= 75).sum()}/{n} ({100*(cov>=75).mean():.1f}%)")

# isna(): kiem tra tung o co rong khong -> sum() dem tong so o rong tren ca 5 cot chinh
thieu = df[["pm25", "temperature", "humidity", "wind", "rain"]].isna().sum().sum()
print(f"Tong so o bi thieu              : {thieu}")

q1, q3 = df.pm25.quantile([.25, .75])   # quantile(): tim gia tri tai dung mot phan tram cho truoc
iqr = q3 - q1
lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
n_out = ((df.pm25 < lo) | (df.pm25 > hi)).sum()   # dem so dong PM2.5 nam ngoai [lo, hi]
print(f"Nguong binh thuong (IQR)        : [{lo:.1f}, {hi:.1f}]")
print(f"So ngay ngoai lai               : {n_out}/{n} ({100*n_out/n:.1f}%)")
print("=> Du lieu day du, it khuyet, ty le ngoai lai thap la dieu tot")

[2] CHAT LUONG DO LUONG
------------------------------------------------------------
Do phu TB (% gio/ngay co do)   : 97.8%
So ngay phu >= 75%             : 615/629 (97.8%)
Tong so o bi thieu              : 0
Nguong binh thuong (IQR)        : [3.2, 60.3]
So ngay ngoai lai               : 18/629 (2.9%)
=> Du lieu day du, it khuyet, ty le ngoai lai thap la dieu tot


## Bằng chứng 3: Cảm biến có hoạt động đúng

**Câu hỏi trả lời:** không phải "data có đủ" mà là "data có ĐÚNG QUY LUẬT VẬT LÝ không" — nếu cảm biến hỏng, số liệu ra ngẫu nhiên, các mối quan hệ vật lý quen thuộc sẽ biến mất.

**2 quy luật kiểm tra:**
1. Nhiệt độ càng cao → độ ẩm càng thấp (trời nóng thì khô) → tương quan phải **ÂM**
2. Gió càng mạnh → bụi càng ít (gió thổi bay bụi) → tương quan phải **ÂM**

Nếu 2 tương quan này đúng chiều (âm) → cảm biến phản ánh đúng thực tế, đáng tin.

In [5]:
print("[3] NHAT QUAN NOI TAI (quy luat vat ly)")
print("-" * 60)

r_th = df.temperature.corr(df.humidity)   # corr(): tinh he so tuong quan Pearson giua 2 cot
print(f"Tuong quan nhiet do - do am : r = {r_th:.3f}   "
      f"{'DUNG (am, dung quy luat)' if r_th < 0 else 'NGHI NGO!'}")

r_wind = df.pm25.corr(df.wind)
print(f"Tuong quan PM2.5 - gio      : r = {r_wind:.3f}   "
      f"{'DUNG (am, dung quy luat)' if r_wind < 0 else 'NGHI NGO!'}")

print("=> Ca 2 dung chieu vat ly -> cam bien hoat dong on dinh, khong phai so rac")

[3] NHAT QUAN NOI TAI (quy luat vat ly)
------------------------------------------------------------
Tuong quan nhiet do - do am : r = -0.486   DUNG (am, dung quy luat)
Tuong quan PM2.5 - gio      : r = -0.420   DUNG (am, dung quy luat)
=> Ca 2 dung chieu vat ly -> cam bien hoat dong on dinh, khong phai so rac


## Bằng chứng 4: Đối chiếu nguồn ĐỘC LẬP (mô hình CAMS)

**Câu hỏi trả lời:** có nguồn nào KHÁC (không liên quan gì tới OpenAQ) đo cùng vị trí, cùng thời gian, cho ra con số gần giống không?

Đây là bước mạnh nhất, vì nếu 2 nguồn **hoàn toàn độc lập** (khác tổ chức, khác công nghệ đo) mà vẫn khớp nhau — khả năng cả 2 cùng sai là rất thấp. Nguồn đối chiếu ở đây là **CAMS** (Copernicus Atmosphere Monitoring Service — mô hình chất lượng không khí của châu Âu), gọi qua chính Open-Meteo, cùng toạ độ trạm CMT8.


In [6]:
print("[4] DOI CHIEU NGUON DOC LAP (CAMS)")
print("-" * 60)
print("Dang tai du lieu CAMS de so sanh...")

r = requests.get(
    "https://air-quality-api.open-meteo.com/v1/air-quality",
    params={
        "latitude": LAT, "longitude": LON,
        "start_date": str(df["date"].min().date()),
        "end_date": str(df["date"].max().date()),
        "hourly": "pm2_5",          # CAMS tra theo GIO -> phai gop ve ngay o duoi
        "timezone": "Asia/Bangkok",
    },
    timeout=90,
)
h = r.json()["hourly"]
cams = pd.DataFrame({"dt": pd.to_datetime(h["time"]), "pm25_cams": h["pm2_5"]})
cams["date"] = cams["dt"].dt.normalize()          # bo phan gio, chi giu ngay
camsd = cams.groupby("date", as_index=False)["pm25_cams"].mean()   # gom 24 gio/ngay -> 1 trung binh/ngay

j = df[["date", "pm25"]].merge(camsd, on="date", how="inner").dropna()
r_cams = j["pm25"].corr(j["pm25_cams"])
tb_tram, tb_cams = j["pm25"].mean(), j["pm25_cams"].mean()
lech = 100 * (tb_tram - tb_cams) / tb_cams

print(f"So ngay doi chieu duoc          : {len(j)}")
print(f"TB tram CMT8 (do that)          : {tb_tram:.2f} ug/m3")
print(f"TB mo hinh CAMS                 : {tb_cams:.2f} ug/m3")
print(f"Chenh lech muc TB               : {lech:+.1f}%  "
      f"{'-> RAT KHOP' if abs(lech) < 15 else '-> lech dang ke'}")
print(f"Tuong quan ngay-theo-ngay       : r = {r_cams:.3f}")
print()
print("DIEN GIAI TRUNG THUC:")
print("  + Muc TRUNG BINH khop tot giua 2 nguon doc lap -> ung ho do lon cua so lieu")
print("  - Tuong quan TUNG NGAY yeu hon, vi CAMS la mo hinh toan cau luoi tho ~40km,")
print("    khong bat duoc bien dong cuc bo cua 1 diem do sat duong pho")

j.to_csv("../01_Data/kiem_chung_CMT8_vs_CAMS.csv", index=False)

[4] DOI CHIEU NGUON DOC LAP (CAMS)
------------------------------------------------------------
Dang tai du lieu CAMS de so sanh...
So ngay doi chieu duoc          : 629
TB tram CMT8 (do that)          : 32.41 ug/m3
TB mo hinh CAMS                 : 29.93 ug/m3
Chenh lech muc TB               : +8.3%  -> RAT KHOP
Tuong quan ngay-theo-ngay       : r = 0.297

DIEN GIAI TRUNG THUC:
  + Muc TRUNG BINH khop tot giua 2 nguon doc lap -> ung ho do lon cua so lieu
  - Tuong quan TUNG NGAY yeu hon, vi CAMS la mo hinh toan cau luoi tho ~40km,
    khong bat duoc bien dong cuc bo cua 1 diem do sat duong pho


## Bằng chứng 5: Khớp với tài liệu khoa học đã công bố

**Câu hỏi trả lời:** kết quả có đi ngược lại những gì nghiên cứu khác đã chứng minh không?

**Tài liệu đối chiếu cụ thể (có DOI, tra được thật):**

1. To Thi Hien và cộng sự (2019), *"Current Status of Fine Particulate Matter (PM2.5) in Vietnam's Most Populous City, Ho Chi Minh City"*, **Aerosol and Air Quality Research**, tập 19, tr. 2239–2251.
   DOI: [10.4209/aaqr.2018.12.0471](https://doi.org/10.4209/aaqr.2018.12.0471)
   — Đo 5 điểm tại TP.HCM (03/2017–03/2018), PM2.5 trung bình năm **36.3 ± 13.7 µg/m³**, vượt quy chuẩn VN (25) lẫn khuyến nghị WHO.

2. *"Annual Variation of PM2.5 Chemical Composition in Ho Chi Minh City, Vietnam Including the COVID-19 Outbreak Period"*, **Aerosol and Air Quality Research**.
   DOI: [10.4209/aaqr.220312](https://doi.org/10.4209/aaqr.220312)
   — Chứng minh PM2.5 **mùa khô cao hơn mùa mưa**, do cơ chế "rửa trôi" (wet deposition/washout): mưa cuốn trôi bụi khỏi không khí, còn mùa khô không có cơ chế này + hiện tượng nghịch nhiệt giữ bụi gần mặt đất.

Nếu data của mình cũng cho ra đúng chiều "mùa khô > mùa mưa" và mức PM2.5 cùng cỡ độ lớn (chục µg/m³, vượt chuẩn VN/WHO) → khớp với 2 nghiên cứu này, thêm 1 bằng chứng ủng hộ độ tin cậy.

In [7]:
print("[5] KHOP TAI LIEU KHOA HOC DA CONG BO")
print("-" * 60)
print("Doi chieu voi 2 nghien cuu da cong bo (co DOI, tra duoc that):")
print("  [1] To Thi Hien et al. (2019), Aerosol & Air Quality Research 19:2239-2251")
print("      DOI: 10.4209/aaqr.2018.12.0471")
print("  [2] Annual Variation of PM2.5 Chemical Composition in HCMC (incl. COVID-19)")
print("      Aerosol & Air Quality Research, DOI: 10.4209/aaqr.220312")
print()

# loc rieng cac dong mua kho / mua mua roi tinh trung binh PM2.5 cua tung nhom
kho = df[df.mua == "Mua kho"].pm25.mean()
mua = df[df.mua == "Mua mua"].pm25.mean()
print(f"PM2.5 TB mua kho (du lieu cua minh) : {kho:.1f} ug/m3")
print(f"PM2.5 TB mua mua (du lieu cua minh) : {mua:.1f} ug/m3")
print(f"-> Mua kho {'CAO HON' if kho > mua else 'THAP HON'} mua mua "
      f"({100*(kho-mua)/mua:+.0f}%)")
print("   => Dung chieu voi ca 2 nghien cuu [1][2]: mua kho o nhiem nang hon")
print("      (co che: mua rua troi bui trong mua mua, mua kho khong co co che nay)")
print()

tb = df.pm25.mean()
print(f"PM2.5 trung binh toan bo cua minh = {tb:.1f} ug/m3")
print(f"  So voi nghien cuu [1]: 36.3 ug/m3 (2017-2018) -- cung do lon (chuc ug/m3)")
# so voi 2 nguong tham chieu: WHO (khuyen nghi quoc te) va QCVN (quy chuan VN, long hon)
print(f"  Gap {tb/5:.1f} lan khuyen nghi WHO 2021 (5 ug/m3)")
print(f"  Gap {tb/25:.1f} lan QCVN 05:2023 cua Viet Nam (25 ug/m3)")
print("  -> Phu hop boi canh do thi lon Dong Nam A, khop voi tai lieu da cong bo")

[5] KHOP TAI LIEU KHOA HOC DA CONG BO
------------------------------------------------------------
Doi chieu voi 2 nghien cuu da cong bo (co DOI, tra duoc that):
  [1] To Thi Hien et al. (2019), Aerosol & Air Quality Research 19:2239-2251
      DOI: 10.4209/aaqr.2018.12.0471
  [2] Annual Variation of PM2.5 Chemical Composition in HCMC (incl. COVID-19)
      Aerosol & Air Quality Research, DOI: 10.4209/aaqr.220312

PM2.5 TB mua kho (du lieu cua minh) : 37.2 ug/m3
PM2.5 TB mua mua (du lieu cua minh) : 27.6 ug/m3
-> Mua kho CAO HON mua mua (+35%)
   => Dung chieu voi ca 2 nghien cuu [1][2]: mua kho o nhiem nang hon
      (co che: mua rua troi bui trong mua mua, mua kho khong co co che nay)

PM2.5 trung binh toan bo cua minh = 32.4 ug/m3
  So voi nghien cuu [1]: 36.3 ug/m3 (2017-2018) -- cung do lon (chuc ug/m3)
  Gap 6.5 lan khuyen nghi WHO 2021 (5 ug/m3)
  Gap 1.3 lan QCVN 05:2023 cua Viet Nam (25 ug/m3)
  -> Phu hop boi canh do thi lon Dong Nam A, khop voi tai lieu da cong bo


## Bằng chứng 6: Nêu rõ hạn chế 

In [8]:
# Day la phan CHU DONG neu ra diem yeu cua chinh du lieu minh dung,
# khong phai code tinh toan gi -> muc dich la trung thuc khoa hoc
print("[6] HAN CHE CUA DU LIEU (phai ghi ro trong bao cao)")
print("-" * 60)
print("a) Tram CMT8 dung cam bien CHI PHI THAP (AirGradient), khong phai")
print("   tram tham chieu chuan quoc gia -> gia tri tuyet doi co the lech.")
print("b) Khong doi chieu duoc voi tram chuan: tram Lanh su quan My (ID 7440)")
print("   da ngung phat so lieu tu 02/11/2023 -> khong trung ky voi du lieu nay.")
print("c) Sensor PM10 cua tram da ngung hoat dong tu 14/01/2025 (da chung minh")
print("   bang code o notebook 00) -> de tai chi dung duoc PM2.5.")

# Han che (d): kiem tra bang so lieu that thay vi noi suong
do_am_tb = df["humidity"].mean()
print(f"d) Do am do duoc (TB {do_am_tb:.1f}%) THAP HON do am ngoai troi TP.HCM")
print("   thuong gap (~70-80% theo so lieu khi tuong) -> co the cam bien dat noi")
print("   che chan. Dung do am nhu bien THAM KHAO, khong ket luan tuyet doi.")

print("e) Du lieu 1 DIEM DO, khong dai dien toan TP.HCM.")
print("=> Neu ro han che la TU DUY KHOA HOC NGHIEM TUC, khong phai diem yeu")

[6] HAN CHE CUA DU LIEU (phai ghi ro trong bao cao)
------------------------------------------------------------
a) Tram CMT8 dung cam bien CHI PHI THAP (AirGradient), khong phai
   tram tham chieu chuan quoc gia -> gia tri tuyet doi co the lech.
b) Khong doi chieu duoc voi tram chuan: tram Lanh su quan My (ID 7440)
   da ngung phat so lieu tu 02/11/2023 -> khong trung ky voi du lieu nay.
c) Sensor PM10 cua tram da ngung hoat dong tu 14/01/2025 (da chung minh
   bang code o notebook 00) -> de tai chi dung duoc PM2.5.
d) Do am do duoc (TB 54.7%) THAP HON do am ngoai troi TP.HCM
   thuong gap (~70-80% theo so lieu khi tuong) -> co the cam bien dat noi
   che chan. Dung do am nhu bien THAM KHAO, khong ket luan tuyet doi.
e) Du lieu 1 DIEM DO, khong dai dien toan TP.HCM.
=> Neu ro han che la TU DUY KHOA HOC NGHIEM TUC, khong phai diem yeu


## Bảng tổng kết — lưu lại để đưa vào báo cáo

In [9]:
# Xay 1 bang tom tat: moi dong la 1 tieu chi, gom ten tieu chi - bang chung - ket qua
# Day chinh la bang de chup man hinh dua vao bao cao
bang = pd.DataFrame([
    ["1. Nguon goc minh bach", "OpenAQ + Open-Meteo, ID tram cong khai", "DAT"],
    ["2. Tai lap duoc", "Notebook day du, chay lai ra cung ket qua", "DAT"],
    ["3. Do phu du lieu", f"{df['pm25_cov'].mean():.1f}% trung binh", "DAT"],
    ["4. Gia tri thieu", f"{thieu} o", "DAT"],
    ["5. Ngoai lai", f"{100*n_out/n:.1f}%", "DAT"],
    ["6. Nhat quan vat ly", f"r(nhiet,am)={r_th:.2f}; r(pm25,gio)={r_wind:.2f}", "DAT"],
    ["7. Doi chieu doc lap", f"Muc TB khop CAMS, lech {lech:+.1f}%", "DAT (1 phan)"],
    ["8. Khop tai lieu KH", "Mua kho > mua mua", "DAT"],
    ["9. Neu ro han che", "5 han che da liet ke", "DAT"],
], columns=["Tieu chi", "Bang chung", "Ket qua"])

bang.to_csv("../01_Data/bang_kiem_chung.csv", index=False)
print("Da luu: ../01_Data/bang_kiem_chung.csv")
bang

Da luu: ../01_Data/bang_kiem_chung.csv


,Tieu chi,Bang chung,Ket qua
0,1. Nguon goc minh bach,"OpenAQ + Open-Meteo, ID tram cong khai",DAT
1,2. Tai lap duoc,"Notebook day du, chay lai ra cung ket qua",DAT
2,3. Do phu du lieu,97.8% trung binh,DAT
3,4. Gia tri thieu,0 o,DAT
4,5. Ngoai lai,2.9%,DAT
5,6. Nhat quan vat ly,"r(nhiet,am)=-0.49; r(pm25,gio)=-0.42",DAT
6,7. Doi chieu doc lap,"Muc TB khop CAMS, lech +8.3%",DAT (1 phan)
7,8. Khop tai lieu KH,Mua kho > mua mua,DAT
8,9. Neu ro han che,5 han che da liet ke,DAT


## Xong Bước 2 — chụp màn hình gì cho báo cáo?

Chụp lại: kết quả 6 ô Trụ cột 1-6 (đặc biệt là Trụ cột 4 - đối chiếu CAMS, và bảng tổng kết cuối)

**Tiếp theo (GĐ 2 — việc của bạn):** mở `02_thong_ke_mo_ta.ipynb` — Thống kê mô tả (mean, median, độ lệch chuẩn, skewness...) trên PM2.5 và các biến X. Báo mình khi sẵn sàng, mình dựng khung notebook đó.